# Chi-Square + Malicious Flow Integration (CIC18 → CIC17)

This notebook evaluates the impact of integrating **malicious (D)DoS network flows** from the target dataset (**GenIDS-CIC18**) into the source dataset (**GenIDS-CIC17**) before training a Machine Learning-based Intrusion Detection System (IDS).

The experiment applies **Chi-Square feature selection** before training the model.

## Experimental strategy

1. Load GenIDS-CIC17 and GenIDS-CIC18.
2. Remove non-feature columns and harmonize the datasets.
3. Encode categorical attributes.
4. Select a percentage of malicious (D)DoS flows from CIC18 for integration.
5. Remove the selected CIC18 flows from the CIC18 generalization test set to avoid data leakage.
6. Remove malicious (D)DoS flows from CIC17 to keep the training set size controlled.
7. Integrate the selected malicious CIC18 flows into CIC17.
8. Apply Min-Max scaling.
9. Apply Chi-Square feature selection.
10. Train an XGBoost classifier.
11. Evaluate the model under:
    - **Intraset scenario**: CIC17 train/test split.
    - **Interset scenario**: CIC18 generalization test.


## 1. Imports and Global Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)

from xgboost import XGBClassifier

In [ ]:
# -------------------------------------------------------------------------
# Experiment configuration
# -------------------------------------------------------------------------

RANDOM_STATE = 42

# Change this value to reproduce the other integration rates: 0.20, 0.40, 0.60, or 0.80
INTEGRATION_RATE = 0.40

# Number of features selected by Chi-Square
N_SELECTED_FEATURES = 25

# Dataset paths
DATA_DIR = Path("/home/kcarvalho/datasets")

SOURCE_DATASET_PATH = DATA_DIR / "GenIDS-CIC17.csv"
TARGET_DATASET_PATH = DATA_DIR / "GenIDS-CIC18.csv"

# Label convention
BENIGN_LABEL = 0
MALICIOUS_LABEL = 1

print(f"Integration rate: {INTEGRATION_RATE:.0%}")
print(f"Number of selected features: {N_SELECTED_FEATURES}")

## 2. Helper Functions

In [ ]:
def load_dataset(file_path: Path) -> pd.DataFrame:
    """Load a dataset from a CSV file."""
    return pd.read_csv(file_path)


def show_class_distribution(df: pd.DataFrame, label_col: str = "binary", title: str = "Dataset") -> None:
    """Print absolute and relative class distributions."""
    print(f"\n{title} - absolute distribution:")
    print(df[label_col].value_counts())

    print(f"\n{title} - relative distribution:")
    print(df[label_col].value_counts(normalize=True).map("{:.2%}".format))


def drop_non_feature_columns(df: pd.DataFrame, columns_to_drop: list[str]) -> pd.DataFrame:
    """Remove columns that should not be used as predictive features."""
    existing_columns = [col for col in columns_to_drop if col in df.columns]
    return df.drop(columns=existing_columns)


def encode_categorical_columns(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    categorical_columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Encode categorical columns using a shared mapping across source and target datasets.
    This avoids inconsistent encodings between datasets.
    """
    source_df = source_df.copy()
    target_df = target_df.copy()

    for column in categorical_columns:
        if column in source_df.columns and column in target_df.columns:
            encoder = LabelEncoder()
            combined_values = pd.concat(
                [source_df[column].astype(str), target_df[column].astype(str)],
                axis=0,
                ignore_index=True,
            )
            encoder.fit(combined_values)

            source_df[column] = encoder.transform(source_df[column].astype(str))
            target_df[column] = encoder.transform(target_df[column].astype(str))

    return source_df, target_df


def standardize_numeric_dtypes(df: pd.DataFrame, label_col: str = "binary") -> pd.DataFrame:
    """Convert numeric feature columns to float64 and the label column to int64."""
    df = df.copy()

    for column in df.columns:
        if column == label_col:
            df[column] = df[column].astype(np.int64)
        elif pd.api.types.is_numeric_dtype(df[column]):
            df[column] = df[column].astype(np.float64)

    return df


def select_class_subset(
    df: pd.DataFrame,
    label_value: int,
    fraction: float,
    label_col: str = "binary",
) -> pd.DataFrame:
    """Select the first fraction of flows belonging to a specific class."""
    class_flows = df[df[label_col] == label_value]
    n_selected = int(len(class_flows) * fraction)
    return class_flows.iloc[:n_selected].copy()


def integrate_flows_by_timestamp(
    source_df: pd.DataFrame,
    target_subset: pd.DataFrame,
    source_name: str,
    target_name: str,
    timestamp_col: str = "bidirectional_first_seen_ms",
) -> pd.DataFrame:
    """Integrate selected target flows into the source dataset and sort by timestamp."""
    source_df = source_df.copy()
    target_subset = target_subset.copy()

    source_df["source_dataset"] = source_name
    target_subset["source_dataset"] = target_name

    integrated_df = pd.concat([source_df, target_subset], ignore_index=True)

    if timestamp_col in integrated_df.columns:
        integrated_df = integrated_df.sort_values(by=timestamp_col)

    integrated_df = integrated_df.drop(columns=["source_dataset"])
    return integrated_df


def evaluate_binary_classifier(
    model,
    X,
    y,
    dataset_name: str,
    positive_label: int = 1,
) -> dict:
    """Evaluate a binary classifier and display standard IDS metrics and curves."""
    y_pred = model.predict(X)
    y_score = model.predict_proba(X)[:, 1]

    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y, y_pred),
        "precision_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y, y_pred, average="macro", zero_division=0),
        "auc_roc": roc_auc_score(y, y_score),
        "auc_pr": average_precision_score(y, y_score, pos_label=positive_label),
        "far": fp / (fp + tn) if (fp + tn) > 0 else 0.0,
    }

    print(f"\n=== {dataset_name} Evaluation ===")
    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y, y_pred, digits=4, zero_division=0))

    for key, value in metrics.items():
        if key != "dataset":
            print(f"{key}: {value:.4f}")

    ConfusionMatrixDisplay(confusion_matrix=cm).plot()
    plt.title(f"Confusion Matrix - {dataset_name}")
    plt.show()

    fpr, tpr, _ = roc_curve(y, y_score)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"AUC-ROC = {metrics['auc_roc']:.4f}")
    plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {dataset_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    precision_values, recall_values, _ = precision_recall_curve(y, y_score, pos_label=positive_label)
    baseline = np.sum(y == positive_label) / len(y)

    plt.figure(figsize=(8, 6))
    plt.plot(recall_values, precision_values, label=f"AUC-PR = {metrics['auc_pr']:.4f}")
    plt.plot([0, 1], [baseline, baseline], "k--", label=f"Baseline = {baseline:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve - {dataset_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return metrics

## 3. Load Datasets

In [ ]:
source_df = load_dataset(SOURCE_DATASET_PATH)
target_df = load_dataset(TARGET_DATASET_PATH)

print("Source dataset shape:", source_df.shape)
print("Target dataset shape:", target_df.shape)

In [ ]:
# In the original CIC18 file, the binary label is stored in the 'label' column.
if "binary" not in target_df.columns and "label" in target_df.columns:
    target_df["binary"] = target_df["label"].copy()

show_class_distribution(source_df, label_col="binary", title="Source dataset - CIC17")
show_class_distribution(target_df, label_col="binary", title="Target dataset - CIC18")

## 4. Preprocessing and Feature Harmonization

In [ ]:
SOURCE_COLUMNS_TO_DROP = [
    "multiclass",
    "mapped_label",
    "date",
    "hours",
    "expiration_id",
    "src_ip",
    "src_mac",
    "src_oui",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "ip_version",
    "vlan_id",
    "tunnel_id",
]

TARGET_COLUMNS_TO_DROP = [
    "multiclass",
    "label",
    "Timestamp",
    "src_ip",
    "src_mac",
    "src_oui",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "ip_version",
    "vlan_id",
    "tunnel_id",
]

source_df = drop_non_feature_columns(source_df, SOURCE_COLUMNS_TO_DROP)
target_df = drop_non_feature_columns(target_df, TARGET_COLUMNS_TO_DROP)

print("Source dataset shape after column removal:", source_df.shape)
print("Target dataset shape after column removal:", target_df.shape)

In [ ]:
CATEGORICAL_COLUMNS = ["application_name", "application_category_name"]

source_df, target_df = encode_categorical_columns(
    source_df=source_df,
    target_df=target_df,
    categorical_columns=CATEGORICAL_COLUMNS,
)

source_df = standardize_numeric_dtypes(source_df, label_col="binary")
target_df = standardize_numeric_dtypes(target_df, label_col="binary")

print(source_df.dtypes.value_counts())
print(target_df.dtypes.value_counts())

In [ ]:
# Ensure that both datasets have the same columns and column order.
common_columns = source_df.columns.intersection(target_df.columns)

source_df = source_df[common_columns].copy()
target_df = target_df[common_columns].copy()

print("Number of common columns:", len(common_columns))
print("Source dataset shape:", source_df.shape)
print("Target dataset shape:", target_df.shape)

## 5. Malicious Flow Selection and Data Leakage Prevention

In [ ]:
# Select malicious (D)DoS flows from the target dataset to be integrated into the source dataset.
target_malicious_subset = select_class_subset(
    df=target_df,
    label_value=MALICIOUS_LABEL,
    fraction=INTEGRATION_RATE,
    label_col="binary",
)

print("Selected malicious flows from target dataset:", target_malicious_subset.shape)
print(target_malicious_subset["binary"].value_counts())

In [ ]:
# Remove the integrated target flows from the target test set to avoid data leakage.
target_test_df = target_df.drop(index=target_malicious_subset.index).copy()

print("Target test dataset after removing integrated flows:", target_test_df.shape)
show_class_distribution(target_test_df, label_col="binary", title="Target test dataset - CIC18")

In [ ]:
# Remove malicious flows from the source dataset to keep the source dataset size controlled.
source_malicious_subset = select_class_subset(
    df=source_df,
    label_value=MALICIOUS_LABEL,
    fraction=INTEGRATION_RATE,
    label_col="binary",
)

source_base_df = source_df.drop(index=source_malicious_subset.index).copy()

print("Removed malicious flows from source dataset:", source_malicious_subset.shape)
print("Source dataset after removal:", source_base_df.shape)
show_class_distribution(source_base_df, label_col="binary", title="Source base dataset - CIC17")

## 6. Flow Integration

In [ ]:
integrated_source_df = integrate_flows_by_timestamp(
    source_df=source_base_df,
    target_subset=target_malicious_subset,
    source_name="cic17",
    target_name=f"cic18_{int(INTEGRATION_RATE * 100)}_malicious",
    timestamp_col="bidirectional_first_seen_ms",
)

print("Integrated source dataset shape:", integrated_source_df.shape)
show_class_distribution(integrated_source_df, label_col="binary", title="Integrated source dataset - CIC17 + CIC18 malicious flows")

## 7. Train/Test Split

In [ ]:
X_source = integrated_source_df.drop(columns=["binary"])
y_source = integrated_source_df["binary"]

X_target = target_test_df.drop(columns=["binary"])
y_target = target_test_df["binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X_source,
    y_source,
    test_size=0.8,
    random_state=RANDOM_STATE,
    stratify=y_source,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_target:", X_target.shape)

## 8. Min-Max Scaling

In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_target_scaled = scaler.transform(X_target)

print("Scaled training set:", X_train_scaled.shape)
print("Scaled intraset test set:", X_test_scaled.shape)
print("Scaled interset test set:", X_target_scaled.shape)

## 9. Chi-Square Feature Selection

In [ ]:
chi2_selector = SelectKBest(score_func=chi2, k=N_SELECTED_FEATURES)

X_train_selected = chi2_selector.fit_transform(X_train_scaled, y_train)
X_test_selected = chi2_selector.transform(X_test_scaled)
X_target_selected = chi2_selector.transform(X_target_scaled)

selected_indices = chi2_selector.get_support(indices=True)
chi2_scores = chi2_selector.scores_

selected_feature_names = [X_train.columns[i] for i in selected_indices]
selected_scores = [chi2_scores[i] for i in selected_indices]

selected_features_df = (
    pd.DataFrame({
        "feature": selected_feature_names,
        "chi2_score": selected_scores,
    })
    .sort_values(by="chi2_score", ascending=False)
    .reset_index(drop=True)
)

print("Selected features:")
display(selected_features_df)

print("X_train_selected:", X_train_selected.shape)
print("X_test_selected:", X_test_selected.shape)
print("X_target_selected:", X_target_selected.shape)

In [ ]:
plt.figure(figsize=(10, 6))
top_features = selected_features_df.head(10).sort_values(by="chi2_score", ascending=True)

plt.barh(top_features["feature"], top_features["chi2_score"])
plt.xlabel("Chi-Square Score")
plt.ylabel("Feature")
plt.title("Top 10 Selected Features by Chi-Square Score")
plt.tight_layout()
plt.show()

## 10. Model Training

In [ ]:
model = XGBClassifier(
    eval_metric="logloss",
    n_estimators=300,
    max_depth=10,
    objective="binary:logistic",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.5,
    random_state=RANDOM_STATE,
)

model.fit(X_train_selected, y_train)

## 11. Intraset Evaluation: CIC17

In [ ]:
intraset_metrics = evaluate_binary_classifier(
    model=model,
    X=X_test_selected,
    y=y_test,
    dataset_name="Intraset - CIC17",
    positive_label=MALICIOUS_LABEL,
)

## 12. Interset Evaluation: CIC18

In [ ]:
interset_metrics = evaluate_binary_classifier(
    model=model,
    X=X_target_selected,
    y=y_target,
    dataset_name="Interset - CIC18",
    positive_label=MALICIOUS_LABEL,
)

## 13. Summary of Results

In [ ]:
results_df = pd.DataFrame([intraset_metrics, interset_metrics])
display(results_df)

## Notes

- The Chi-Square selector is fitted only on the training subset to avoid data leakage.
- The scaler is fitted only on the training subset and then applied to both intraset and interset test sets.
- Integrated CIC18 malicious flows are removed from the CIC18 test set before evaluation.
- This notebook can be reused for 20%, 40%, 60%, and 80% integration scenarios by changing `INTEGRATION_RATE`.
